# 2층 신경망의 역전파
입력 2개 → 은닉 2개 → 출력 1개. 배치 하나를 열벡터 규약으로 계산한다. 배열 shape는 NumPy 기준으로 적으며 스칼라는 `()`로 표기한다.

원본 3쪽에서 비어 있는 레이블은 7쪽의 예시를 따라 $y=1$로 둔다. 모든 중간값은 가능한 자릿수를 유지하고, 보고할 때만 소수점 4자리로 반올림한다.

## 1. 순전파 손계산

| 기호 | shape | 입력 또는 계산 |
| --- | --- | --- |
| $x$ | `(2,)` | $[1,0]$ |
| $W_1$ | `(2,2)` | $[[0.1,0.2],[0.3,0.4]]$ |
| $b_1$ | `(2,)` | $[0,0]$ |
| $W_2$ | `(2,)` | $[0.5,0.6]$ |
| $b_2$, $y$ | `()` | $0$, $1$ |
| $z_1$ | `(2,)` | $W_1x+b_1=[0.1\cdot1+0.2\cdot0,\;0.3\cdot1+0.4\cdot0]=[0.1,0.3]$ |
| $a_1$ | `(2,)` | $\sigma(z_1) \approx [0.52497919,0.57444252]$ |
| $z_2$ | `()` | $W_2^\top a_1+b_2=0.5(0.52497919)+0.6(0.57444252)\approx0.60715510$ |
| $\hat y$ | `()` | $\sigma(z_2)=1/(1+e^{-z_2})\approx0.64729157$ |
| $L$ | `()` | $-[y\log\hat y+(1-y)\log(1-\hat y)]=-\log(0.64729157)\approx0.43495844$ |

첫 층 가중치 행렬의 각 행은 은닉 뉴런 하나에 대응한다. 그 행의 두 가중치를 입력 두 성분에 각각 곱해 더하고, 해당 뉴런의 편향을 더한다. 출력층에서는 은닉 활성값 2개를 가중합해 확률 하나로 바꾼다.

## 2. 연쇄 법칙으로 거꾸로 미분

손실에 가까운 변수를 먼저 미분한다. $\odot$는 같은 위치끼리 곱하기, $\delta_2=\partial L/\partial z_2$, $\delta_1=\partial L/\partial z_1$이다.

1. **확률 $\hat y$에 대한 기울기** `()`  
   $\displaystyle \frac{\partial L}{\partial\hat y}=-\frac{y}{\hat y}+\frac{1-y}{1-\hat y}\approx-1/0.64729157=-1.54489885$.
2. **로짓 $z_2$** `()`  
   $\displaystyle \delta_2=\frac{\partial L}{\partial\hat y}\hat y(1-\hat y)=-y(1-\hat y)+(1-y)\hat y=\hat y-y\approx-0.35270843$. BCE와 sigmoid의 미분이 곱해져 간단해진다.
3. **출력층 가중치 $W_2$** `(2,)`, **편향 $b_2$** `()`  
   $z_2=\sum_j W_{2,j}a_{1,j}+b_2$이므로 $\partial z_2/\partial W_{2,j}=a_{1,j}$이다. $\nabla_{W_2}L=\delta_2 a_1\approx[-0.18516459,-0.20261072]$, $\partial L/\partial b_2=\delta_2$.
4. **은닉 활성값 $a_1$** `(2,)`  
   $\partial z_2/\partial a_{1,j}=W_{2,j}$이므로 $\nabla_{a_1}L=\delta_2 W_2=(-0.35270843)[0.5,0.6]\approx[-0.17635422,-0.21162506]$.
5. **은닉 로짓 $z_1$** `(2,)`  
   각 sigmoid는 해당 성분만 사용한다. $a_1\odot(1-a_1)\approx[0.24937604,0.24445831]$이므로 $\delta_1=\nabla_{a_1}L\odot a_1\odot(1-a_1)\approx[-0.04397852,-0.05173350]$.
6. **첫 층 가중치 $W_1$** `(2,2)`, **편향 $b_1$** `(2,)`  
   $z_{1,i}=\sum_j W_{1,ij}x_j+b_{1,i}$이므로 $\partial L/\partial W_{1,ij}=\delta_{1,i}x_j$. 외적 $\delta_1 x^\top\approx[[-0.04397852,0],[-0.05173350,0]]$이고 $\nabla_{b_1}L=\delta_1$이다. 입력 두 번째 성분이 0이라 해당 열은 이 샘플에서 움직이지 않는다.

원본 7쪽의 표시값 $z_2=0.6071$, $\delta_{1,2}=-0.0513$을 재계산하면 4자리 반올림 기준으로 각각 $0.6072$, $-0.0517$이다.

In [1]:
import numpy as np

np.random.seed(42)

def sigmoid(z):
    """시그모이드 1 / (1 + exp(-z))를 계산한다."""
    return 1 / (1 + np.exp(-z))

x = np.array([1.0, 0.0])
w1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.zeros(2)
w2 = np.array([0.5, 0.6])
b2, y = 0.0, 1.0

z1 = w1 @ x + b1
a1 = sigmoid(z1)
z2 = w2 @ a1 + b2
y_pred = sigmoid(z2)
loss = -(y * np.log(y_pred) + (1 - y) * np.log1p(-y_pred))

d_y = -y / y_pred + (1 - y) / (1 - y_pred)
d_z2 = d_y * y_pred * (1 - y_pred)
np.testing.assert_allclose(d_z2, y_pred - y, atol=1e-14, rtol=0)
d_w2 = d_z2 * a1
d_a1 = d_z2 * w2
d_z1 = d_a1 * a1 * (1 - a1)
d_w1 = np.outer(d_z1, x)
d_b2, d_b1 = d_z2, d_z1

In [2]:
# 앞에서 유도한 4자리 손계산 값을 별도로 입력한다.
expected = {
    "z1": ([0.1, 0.3], z1),
    "a1": ([0.5250, 0.5744], a1),
    "z2": (0.6072, z2),
    "y_pred": (0.6473, y_pred),
    "L": (0.4350, loss),
    "dL/dy_pred": (-1.5449, d_y),
    "dL/dz2": (-0.3527, d_z2),
    "dL/dW2": ([-0.1852, -0.2026], d_w2),
    "dL/da1": ([-0.1764, -0.2116], d_a1),
    "dL/dz1": ([-0.0440, -0.0517], d_z1),
    "dL/dW1": ([[-0.0440, 0], [-0.0517, 0]], d_w1),
    "dL/db1": ([-0.0440, -0.0517], d_b1),
    "dL/db2": (-0.3527, d_b2),
}
for name, (hand, actual) in expected.items():
    np.testing.assert_array_equal(np.round(actual, 4), hand)
    shown = np.array2string(
        np.asarray(actual), formatter={"float_kind": lambda value: f"{value:.4f}"}
    )
    print(f"{name:11s} {str(np.shape(actual)):7s} {shown}")
print(f"Hand calculation vs NumPy: {len(expected)}/{len(expected)} match to 4 decimals [PASS]")

z1          (2,)    [0.1000 0.3000]
a1          (2,)    [0.5250 0.5744]
z2          ()      0.6072
y_pred      ()      0.6473
L           ()      0.4350
dL/dy_pred  ()      -1.5449
dL/dz2      ()      -0.3527
dL/dW2      (2,)    [-0.1852 -0.2026]
dL/da1      (2,)    [-0.1764 -0.2116]
dL/dz1      (2,)    [-0.0440 -0.0517]
dL/dW1      (2, 2)  [[-0.0440 -0.0000]
 [-0.0517 -0.0000]]
dL/db1      (2,)    [-0.0440 -0.0517]
dL/db2      ()      -0.3527
Hand calculation vs NumPy: 13/13 match to 4 decimals [PASS]


## 3. 수식 자체의 독립 검증

위의 13개 일치는 손계산과 NumPy가 같은 역전파 수식에서 나왔다는 것을 보여 준다. 수식이 맞는지는 연쇄 법칙을 쓰지 않는 방법으로 따로 확인한다. 각 파라미터를 $\pm h$만큼 움직여 손실 변화를 직접 재는 중심차분이다.

$$\frac{\partial L}{\partial\theta}\approx\frac{L(\theta+h)-L(\theta-h)}{2h}$$

모든 가중치와 편향에서 역전파 기울기와 중심차분 기울기의 차이가 $10^{-8}$ 이하인지 확인한다.

In [3]:
def loss_from(params):
    """파라미터 묶음 (W1, b1, W2, b2)로 순전파를 다시 계산해 BCE 손실을 반환한다."""
    w1_, b1_, w2_, b2_ = params
    a1_ = sigmoid(w1_ @ x + b1_)
    y_pred_ = sigmoid(w2_ @ a1_ + b2_)
    return -(y * np.log(y_pred_) + (1 - y) * np.log1p(-y_pred_))


def numerical_gradient(index, h=1e-6):
    """params[index]의 각 원소를 ±h 움직여 중심차분 기울기를 구한다.

    ∂L/∂θ ≈ (L(θ + h) - L(θ - h)) / (2h). 연쇄 법칙을 쓰지 않는 독립 검증이다.
    """
    params = [w1.copy(), b1.copy(), w2.copy(), np.array(b2, dtype=float)]
    target = params[index]
    grad = np.zeros_like(target)
    for pos in np.ndindex(target.shape):
        original = target[pos]
        target[pos] = original + h
        plus = loss_from(params)
        target[pos] = original - h
        minus = loss_from(params)
        target[pos] = original
        grad[pos] = (plus - minus) / (2 * h)
    return grad


analytic = {"dL/dW1": d_w1, "dL/db1": d_b1, "dL/dW2": d_w2, "dL/db2": d_b2}
worst = 0.0
for index, (name, value) in enumerate(analytic.items()):
    numeric = numerical_gradient(index)
    gap = float(np.max(np.abs(numeric - value)))
    assert gap <= 1e-8, f"{name}: gradient error {gap:.3e}"
    worst = max(worst, gap)
    print(f"{name:7s} {str(np.shape(value)):7s} max |analytic - numeric| = {gap:.3e}")
print(f"Central-difference gradient check: max error {worst:.3e} [PASS]")

dL/dW1  (2, 2)  max |analytic - numeric| = 1.036e-10
dL/db1  (2,)    max |analytic - numeric| = 1.036e-10
dL/dW2  (2,)    max |analytic - numeric| = 8.556e-11
dL/db2  ()      max |analytic - numeric| = 1.273e-11
Central-difference gradient check: max error 1.036e-10 [PASS]
